# Online Retail | Sales Intelligence & Customer Segmentation

**Portfolio project · Python · Pandas · Seaborn · Plotly**

> **Business challenge:** Transform raw invoice-line transactions into a reproducible analysis of sales, customers, products, countries and cross-selling opportunities.

**Deliverables:** auditable data-quality checks, cleaned merchandise sales data, invoice-level KPIs, RFM customer segments, time/product/country insights, a co-purchase study, interactive HTML dashboard and exported summary tables.

**Data:** [UCI Online Retail](https://archive.ics.uci.edu/dataset/352/online+retail), by Daqing Chen (2015), DOI: [10.24432/C5BW33](https://doi.org/10.24432/C5BW33), CC BY 4.0. UK gift retailer; transactions **1 Dec 2010–9 Dec 2011**. The raw dataset is observational; this is **not** a study of all retailers.

**Definition used here:** *merchandise gross sales* = sum of `Quantity × UnitPrice` for positive, non-cancelled, merchandise product lines. **Returns, discounts, postage and fees are not netted into the metric.** Values are in GBP; gross sales are **not profits**.

*Reproducibility:* Run the cells top-to-bottom. The notebook looks for local data first and otherwise downloads the UCI ZIP to `data/` when online. No hidden spreadsheets or proprietary services are required. Outputs are saved under `outputs/`.


## 1 · Questions, grain and scope

| Question | Grain | Key measure |
|---|---|---|
| How much merchandise was sold? | Invoice | Revenue, number of orders, AOV |
| Which products lead? | Stock code | Revenue, units, invoices |
| When does demand change? | Month / weekday / hour | Sales and order volume |
| Which markets matter? | Invoice country | Sales, orders and AOV |
| Which customers should be prioritized? | Known customer ID | Recency, frequency, monetary value |
| Which products might be bundled? | Invoice basket | Co-occurrence, support and lift |

**Important:** Rows are *invoice lines*, not separate orders. Customer IDs are missing in part of the dataset; do **not** invent customer identities. First/last months contain partial coverage, and large wholesale orders may be genuine rather than errors.


In [1]:
from pathlib import Path
from collections import Counter
from itertools import combinations
from urllib.request import urlretrieve
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid", palette=["#2F6B65", "#D1825F", "#A2B8AE", "#394D5D"])
plt.rcParams.update({"figure.figsize": (10, 5), "axes.titlesize": 13, "axes.titleweight": "bold"})
BASE = Path.cwd()
DATA_DIR = BASE / "data"
OUT = BASE / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)



In [3]:
# We read the ORIGINAL, uncleaned data. Cleaning happens visibly in Section 3 --
# a real analyst should never silently pre-clean before showing you what was wrong.
df_raw = pd.read_csv("online_retail.csv.gz", compression="gzip")

#df_raw = pd.read_csv("online_retail.csv")

# InvoiceDate arrives as text from CSV; we parse it now because almost every later
# feature (Section 4) and quality check (cancellations, ordering) depends on it being
# a real datetime, not a string.
df_raw["InvoiceDate"] = pd.to_datetime(df_raw["InvoiceDate"], errors="coerce")

print(f"Raw shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
df_raw.head()

Raw shape: 541,909 rows x 8 columns


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,"17,850.00",United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,"17,850.00",United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom


## 3 · Data-quality audit and explicit cleaning rules

1. Remove **exact duplicate rows** as a documented assumption; identical legitimate invoice lines could exist, so this decision is revisitable.
2. Separate cancelled invoices (`C...`) and non-positive quantity returns/adjustments from merchandise sales. Do not describe gross sales as net revenue.
3. Exclude non-positive unit prices, missing dates and identifiable non-merchandise charges from the merchandise view; keep a count of every exclusion category.
4. Keep valid unusually large sales; investigate extremes separately rather than silently trimming them.
5. Do **not** fill missing customer IDs from other invoice rows or invent them. Customer analysis covers the identified subset only.

Exclusion flags **overlap**, so their individual counts must not be summed as if they were disjoint.


In [4]:
NON_MERCH_CODES = {
    "POST", "D", "M", "BANK CHARGES", "DOT", "C2", "CRUK",
    "PADS", "AMAZONFEE", "S", "B", "GIFT_0001",
}
flags = pd.DataFrame(index=raw.index)
flags["exact_duplicate"] = raw.duplicated()
flags["cancelled_invoice"] = raw["InvoiceNo"].str.upper().str.startswith("C").fillna(False)
flags["nonpositive_quantity"] = raw["Quantity"].le(0).fillna(True)
flags["nonpositive_price"] = raw["UnitPrice"].le(0).fillna(True)
flags["invalid_date"] = raw["InvoiceDate"].isna()
flags["non_merchandise"] = raw["StockCode"].str.upper().isin(NON_MERCH_CODES)
flags["missing_description"] = raw["Description"].isna() | raw["Description"].eq("")
flags["missing_customer_id"] = raw["CustomerID"].isna()

audit = pd.DataFrame({
    "Issue": list(flags.columns),
    "Rows affected": [int(flags[col].sum()) for col in flags],
})
audit["Share of raw rows (%)"] = audit["Rows affected"] / len(raw) * 100
display(audit.round(2))

# Customer ID missingness is NOT a reason to exclude an otherwise valid sale.
exclude_flags = [c for c in flags.columns if c != "missing_customer_id"]
keep = ~flags[exclude_flags].any(axis=1)
sales = raw.loc[keep].copy()
sales["Revenue_GBP"] = sales["Quantity"] * sales["UnitPrice"]
assert sales["Revenue_GBP"].gt(0).all()
assert sales["InvoiceDate"].notna().all()
print(f"Retained merchandise lines: {len(sales):,} / {len(raw):,}")
print(f"Excluded from merchandise gross-sales view: {len(raw) - len(sales):,} lines")
print(f"Unidentified customer rows retained in sales: {sales['CustomerID'].isna().sum():,}")


NameError: name 'raw' is not defined

## 4 · Analytical tables and KPI definitions

An **order** is a unique `InvoiceNo`, not a row. The invoice view is the basis of AOV and market order counts. Date features are derived only after validating datetime values.


In [ ]:
sales["Month"] = sales["InvoiceDate"].dt.to_period("M").dt.to_timestamp()
sales["Weekday"] = sales["InvoiceDate"].dt.day_name()
sales["Hour"] = sales["InvoiceDate"].dt.hour
sales["Weekend"] = sales["InvoiceDate"].dt.dayofweek >= 5
sales["Product"] = sales["Description"].str.upper()

invoices = (sales.groupby("InvoiceNo", as_index=False)
    .agg(Revenue_GBP=("Revenue_GBP", "sum"),
         Units=("Quantity", "sum"),
         Lines=("StockCode", "size"),
         InvoiceDate=("InvoiceDate", "min"),
         Country=("Country", "first"),
         CustomerID=("CustomerID", "first")))

kpis = pd.Series({
    "Merchandise gross sales (£)": sales["Revenue_GBP"].sum(),
    "Merchandise invoices": len(invoices),
    "Average invoice value (£)": invoices["Revenue_GBP"].mean(),
    "Units sold": sales["Quantity"].sum(),
    "Known customers": sales["CustomerID"].nunique(),
    "Unique stock codes": sales["StockCode"].nunique(),
})
display(kpis.to_frame(name="Value"))


## 5 · Sales seasonality

The first and final month of the dataset are **partial**. Monthly figures below describe *recorded* sales and should not be compared as equivalent full months without normalizing exposure. Hour-of-day is recorded time, not necessarily a customer delivery time.


In [ ]:
monthly = (sales.groupby("Month", as_index=False)
           .agg(Revenue_GBP=("Revenue_GBP", "sum"), Units=("Quantity", "sum")))
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_sales = sales.groupby("Weekday")["Revenue_GBP"].sum().reindex(weekday_order, fill_value=0)
hourly_sales = sales.groupby("Hour")["Revenue_GBP"].sum().reindex(range(24), fill_value=0)

fig, axs = plt.subplots(1, 3, figsize=(17, 4.5))
axs[0].plot(monthly["Month"], monthly["Revenue_GBP"] / 1e6, marker="o", color="#2F6B65")
axs[0].set(title="Monthly sales (partial months present)", ylabel="GBP millions")
axs[0].tick_params(axis="x", labelrotation=50)
axs[1].bar(weekday_sales.index.str[:3], weekday_sales.values / 1e6, color="#D1825F")
axs[1].set(title="Sales by weekday", ylabel="GBP millions")
axs[2].bar(hourly_sales.index, hourly_sales.values / 1e6, color="#446B77")
axs[2].set(title="Sales by invoice hour", xlabel="Hour", ylabel="GBP millions")
plt.tight_layout(); plt.show()


## 6 · Product performance

Use `StockCode` as the product identifier (descriptions sometimes vary), and a representative description only as a display label. Unit volume and sales value answer different questions.


In [ ]:
product_stats = (sales.groupby("StockCode", as_index=False)
                 .agg(Revenue_GBP=("Revenue_GBP", "sum"),
                      Units=("Quantity", "sum"),
                      Orders=("InvoiceNo", "nunique")))
labels = sales.drop_duplicates("StockCode").set_index("StockCode")["Product"]
product_stats["Label"] = product_stats["StockCode"].map(labels).fillna("Unknown product")
top_revenue = product_stats.nlargest(10, "Revenue_GBP")
top_units = product_stats.nlargest(10, "Units")
fig, axs = plt.subplots(1, 2, figsize=(15, 6))
axs[0].barh(top_revenue["Label"].str[:32][::-1], top_revenue["Revenue_GBP"][::-1], color="#2F6B65")
axs[0].set(title="Top products by merchandise sales", xlabel="GBP")
axs[1].barh(top_units["Label"].str[:32][::-1], top_units["Units"][::-1], color="#D1825F")
axs[1].set(title="Top products by units", xlabel="Units")
plt.tight_layout(); plt.show()
print("Overlap between top-10 product codes:", len(set(top_revenue.StockCode) & set(top_units.StockCode)))


## 7 · Market analysis

Compare revenue and order counts by invoice country. AOV uses distinct **invoices per country**, not product-line counts. Very small countries can have volatile AOV, so check order volume before acting.


In [ ]:
markets = (invoices.groupby("Country", as_index=False)
           .agg(Revenue_GBP=("Revenue_GBP", "sum"), Orders=("InvoiceNo", "nunique")))
markets["AOV_GBP"] = markets["Revenue_GBP"] / markets["Orders"]
markets = markets.sort_values("Revenue_GBP", ascending=False)
display(markets.head(12))
non_uk = markets[markets["Country"] != "United Kingdom"].head(10)
plt.figure(figsize=(10, 5))
sns.barplot(data=non_uk, y="Country", x="Revenue_GBP", color="#446B77")
plt.title("Highest-value markets outside the UK"); plt.xlabel("Merchandise sales (£)"); plt.tight_layout(); plt.show()


## 8 · Customer behavior and RFM segmentation

**Recency:** days between last recorded purchase and one day after the final observed transaction. **Frequency:** distinct invoices, not product lines. **Monetary:** observed positive merchandise sales; **not** lifetime profit.

Ties are scored together using average percentile ranks, preventing single-order customers from being arbitrarily split into different frequency quartiles. Segments are practical **heuristics**, not labels from a validated churn model.


In [ ]:
known = sales.dropna(subset=["CustomerID"]).copy()
customers = (known.groupby("CustomerID", as_index=False)
    .agg(LastPurchase=("InvoiceDate", "max"),
         FirstPurchase=("InvoiceDate", "min"),
         Frequency=("InvoiceNo", "nunique"),
         Monetary_GBP=("Revenue_GBP", "sum"),
         Units=("Quantity", "sum")))

snapshot_date = sales["InvoiceDate"].max() + pd.Timedelta(days=1)
customers["RecencyDays"] = (snapshot_date - customers["LastPurchase"]).dt.days
customers["AOV_GBP"] = customers["Monetary_GBP"] / customers["Frequency"]

def quartile_score(series, high_is_good=True):
    # Equal values always receive the same score, even for highly tied frequencies.
    q = np.ceil(series.rank(method="average", pct=True) * 4).clip(1, 4).astype(int)
    return q if high_is_good else 5 - q

customers["R"] = quartile_score(customers["RecencyDays"], high_is_good=False)
customers["F"] = quartile_score(customers["Frequency"])
customers["M"] = quartile_score(customers["Monetary_GBP"])
customers["RFM"] = customers[["R", "F", "M"]].astype(str).agg("".join, axis=1)

def segment_customer(row):
    r, f = int(row["R"]), int(row["F"])
    if r == 4 and f == 4: return "Champions"
    if r >= 3 and f >= 3: return "Loyal"
    if r >= 3 and f <= 2: return "Recent / Low frequency"
    if r <= 2 and f >= 3: return "At risk (heuristic)"
    return "Dormant / Low frequency"

customers["Segment"] = customers.apply(segment_customer, axis=1)
segments = (customers.groupby("Segment", as_index=False)
            .agg(Customers=("CustomerID", "size"), Revenue_GBP=("Monetary_GBP", "sum")))
segments["CustomerSharePct"] = 100 * segments["Customers"] / segments["Customers"].sum()
segments["RevenueSharePct"] = 100 * segments["Revenue_GBP"] / segments["Revenue_GBP"].sum()
segments = segments.sort_values("Revenue_GBP", ascending=False)
display(segments.round(2))
fig, axs = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=segments, x="CustomerSharePct", y="Segment", color="#2F6B65", ax=axs[0])
sns.barplot(data=segments, x="RevenueSharePct", y="Segment", color="#D1825F", ax=axs[1])
axs[0].set(title="Share of identified customers", xlabel="%", ylabel="")
axs[1].set(title="Share of identified-customer sales", xlabel="%", ylabel="")
plt.tight_layout(); plt.show()


## 9 · Customer relationships and concentration

A high-frequency customer often has high monetary value, but this is a description of the sample, not evidence that purchasing frequency *causes* revenue. A log scale makes large wholesale accounts easier to compare without deleting them.


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 5))
axs[0].scatter(customers["Frequency"], customers["Monetary_GBP"], s=14, alpha=.28, color="#2F6B65")
axs[0].set(xlabel="Number of orders", ylabel="Observed customer sales (£)", title="Frequency vs value")
axs[0].set_xscale("log"); axs[0].set_yscale("log")
axs[1].scatter(customers["RecencyDays"], customers["Monetary_GBP"], s=14, alpha=.28, color="#D1825F")
axs[1].set(xlabel="Days since last purchase", ylabel="Observed customer sales (£)", title="Recency vs value")
axs[1].set_yscale("log")
plt.tight_layout(); plt.show()
top_10_customers_share = 100 * customers.nlargest(10, "Monetary_GBP")["Monetary_GBP"].sum() / customers["Monetary_GBP"].sum()
print(f"Top 10 known customers' share of identified-customer merchandise sales: {top_10_customers_share:.1f}%")


## 10 · Co-purchase patterns (candidate bundles)

Count how often pairs of popular **stock codes** appear on the same distinct invoice. **Support** = shared invoices / all merchandise invoices. **Lift** compares observed co-occurrence with what independent purchasing would imply. Use a minimum co-occurrence threshold to reduce noisy pairs.

**Scope:** top 45 products by distinct-invoice frequency; these are exploratory bundle *candidates*, not evidence of incremental sales or a causal cross-sell effect. A/B tests are needed.


In [ ]:
popular_codes = (sales.groupby("StockCode")["InvoiceNo"].nunique()
                 .nlargest(45).index.tolist())
popular_lines = sales.loc[sales["StockCode"].isin(popular_codes), ["InvoiceNo", "StockCode"]].drop_duplicates()
single_counts = popular_lines.groupby("StockCode")["InvoiceNo"].nunique().to_dict()
pair_counts = Counter()
for basket in popular_lines.groupby("InvoiceNo")["StockCode"].agg(list):
    pair_counts.update(combinations(sorted(set(basket)), 2))
n_orders = invoices["InvoiceNo"].nunique()
min_cooccurrence = max(10, int(np.ceil(n_orders * 0.001)))
pairs = []
for (a, b), both in pair_counts.items():
    if both >= min_cooccurrence:
        support = both / n_orders
        lift = support / ((single_counts[a] / n_orders) * (single_counts[b] / n_orders))
        pairs.append((a, b, both, support, lift))
pair_stats = pd.DataFrame(pairs, columns=["ProductA", "ProductB", "SharedInvoices", "Support", "Lift"])
if not pair_stats.empty:
    pair_stats = pair_stats.sort_values(["SharedInvoices", "Lift"], ascending=False)
    pair_stats["ProductA_Name"] = pair_stats["ProductA"].map(labels)
    pair_stats["ProductB_Name"] = pair_stats["ProductB"].map(labels)
    display(pair_stats.head(12))
else:
    print("No popular product pairs met the minimum co-occurrence threshold.")


## 11 · Investigate high-value outliers without deleting them

Bulk purchasing is part of this retailer's business model. This section surfaces the biggest orders and checks their relative weight in the portfolio; it does not automatically winsorize legitimate purchases.


In [ ]:
q99 = invoices["Revenue_GBP"].quantile(0.99)
large_orders = invoices[invoices["Revenue_GBP"] >= q99].sort_values("Revenue_GBP", ascending=False)
top_one_share = 100 * large_orders["Revenue_GBP"].sum() / invoices["Revenue_GBP"].sum()
print(f"99th percentile of invoice value: £{q99:,.2f}")
print(f"Top 1% (by invoice value) account for {top_one_share:.1f}% of recorded merchandise gross sales")
display(large_orders.head(10))


## 12 · Management insights and decisions

Findings below are **calculated at runtime from the loaded data** rather than typed-in figures. Recommendations are proposals to investigate, not demonstrated causal effects.


In [ ]:
leader = top_revenue.iloc[0]
market_leader = non_uk.iloc[0] if len(non_uk) else None
best_segment = segments.iloc[0]
peak_month = monthly.loc[monthly["Revenue_GBP"].idxmax()]
pair_text = (f"{pair_stats.iloc[0]['ProductA_Name']} + {pair_stats.iloc[0]['ProductB_Name']} "
             f"({int(pair_stats.iloc[0]['SharedInvoices']):,} shared invoices)"
             if not pair_stats.empty else "No pair met the defined threshold")

summary = f"""# Online Retail — Executive Summary

**Dataset:** UCI Online Retail, transactions {raw['InvoiceDate'].min():%d %b %Y}–{raw['InvoiceDate'].max():%d %b %Y}.

**Measured performance**
- **£{kpis['Merchandise gross sales (£)']:,.2f}** merchandise gross sales across **{len(invoices):,}** distinct invoices; mean invoice value **£{kpis['Average invoice value (£)']:,.2f}**.
- **{sales['CustomerID'].nunique():,}** identified purchasing customers; unidentified customers are omitted only from customer-level analysis.
- Top product by sales: **{leader['Label']}** (stock code `{leader['StockCode']}`, £{leader['Revenue_GBP']:,.2f}).
- Highest recorded sales month: **{peak_month['Month']:%B %Y}** (£{peak_month['Revenue_GBP']:,.2f}); interpret first/last partial months cautiously.
- Largest identified-customer RFM segment by sales: **{best_segment['Segment']}** ({best_segment['RevenueSharePct']:.1f}% of identified-customer sales; {int(best_segment['Customers']):,} customers).
- Largest market outside the UK: **{market_leader['Country'] if market_leader is not None else 'N/A'}**.
- Leading co-purchase candidate among frequent products: **{pair_text}**.

**Suggested next steps**
1. Pilot targeted retention campaigns by RFM segment and measure uplift against a holdout group.
2. Review high-value outlier invoices with account managers rather than automatically deleting them.
3. Test a candidate bundle with randomized experimentation before claiming incremental revenue.

**Limitations:** gross merchandise sales exclude refunds/discounts/non-merchandise charges; there is no margin or marketing-cost data. Results come from one historical retailer, and no causal inference or future-demand forecasting is claimed.
"""
display(Markdown(summary))
(OUT / "EXECUTIVE_SUMMARY.md").write_text(summary, encoding="utf-8")


## 13 · Interactive, portable dashboard

This **Plotly HTML dashboard** is self-contained (hover, zoom, pan, legend controls) and opens locally in any browser. It requires **no** Dash server, `ipywidgets` or Jupyter runtime after export. It summarizes the same cleaned merchandise sales tables computed above.


In [ ]:
fig = make_subplots(rows=2, cols=2, horizontal_spacing=.20, vertical_spacing=.16,
                    subplot_titles=("Recorded monthly sales", "Top products by revenue",
                                    "Largest non-UK markets", "RFM revenue by segment"))
fig.add_trace(go.Scatter(x=monthly["Month"], y=monthly["Revenue_GBP"], mode="lines+markers",
                         line=dict(color="#2F6B65", width=3), name="Monthly sales", hovertemplate="%{x|%b %Y}<br>£%{y:,.0f}<extra></extra>"), row=1, col=1)
fig.add_trace(go.Bar(x=top_revenue["Label"].str[:26].iloc[::-1], y=top_revenue["Revenue_GBP"].iloc[::-1],
                     marker_color="#D1825F", name="Top products"), row=1, col=2)
fig.add_trace(go.Bar(x=non_uk["Country"], y=non_uk["Revenue_GBP"],
                     marker_color="#446B77", name="Markets"), row=2, col=1)
fig.add_trace(go.Bar(x=segments["Segment"], y=segments["Revenue_GBP"],
                     marker_color="#79A997", name="Segments"), row=2, col=2)
fig.update_xaxes(tickangle=-45, row=1, col=2)
fig.update_xaxes(tickangle=-35, row=2, col=1)
fig.update_xaxes(tickangle=-35, row=2, col=2)
fig.update_yaxes(tickprefix="£", tickformat="~s")
fig.update_layout(
    template="plotly_white", paper_bgcolor="#F5F2EC", plot_bgcolor="#FFFFFF",
    height=930, showlegend=False, title=dict(text="ONLINE RETAIL | SALES INTELLIGENCE", font=dict(size=25, color="#203B38")),
    margin=dict(t=120, l=80, r=55, b=120),
    annotations=[*fig.layout.annotations, dict(
        x=.5, y=1.085, xref="paper", yref="paper", showarrow=False,
        text=f"Gross merchandise sales £{sales['Revenue_GBP'].sum():,.0f}   •   {len(invoices):,} orders   •   AOV £{invoices['Revenue_GBP'].mean():,.0f}",
        font=dict(size=13, color="#526866"))]
)
dashboard_file = OUT / "online_retail_dashboard.html"
fig.write_html(dashboard_file, include_plotlyjs=True, full_html=True)
print("Saved standalone interactive dashboard:", dashboard_file)
fig.show()


## 14 · Export tidy tables and reproducibility checks

Publish the notebook and `outputs/` tables with their source attribution. For large datasets, keeping raw data out of GitHub is reasonable; the data load cell downloads the official source on demand.


In [ ]:
exports = {
    "quality_audit.csv": audit,
    "invoices.csv": invoices,
    "monthly_sales.csv": monthly,
    "product_performance.csv": product_stats,
    "country_performance.csv": markets,
    "customer_segments.csv": customers.drop(columns=["LastPurchase", "FirstPurchase"]),
    "segment_summary.csv": segments,
    "copurchase_candidates.csv": pair_stats,
}
for name, table in exports.items():
    table.to_csv(OUT / name, index=False)
assert invoices["Revenue_GBP"].sum() == sales["Revenue_GBP"].sum()
assert invoices["InvoiceNo"].is_unique
assert (invoices["Revenue_GBP"] > 0).all()
assert customers["CustomerID"].is_unique
print(f"PASS: {len(exports)} CSV outputs saved; invoice and customer grains validated.")
